# Calculo de kernels

Este notebook evalua los seis kernels KDE con la grilla y los bandwidths elegidos.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity

cwd = Path.cwd().resolve()
notebook_dir = cwd if cwd.name == "Kernel_Tests" else cwd / "Kernel_Tests"


def resolve_data_path(relative_path):
    data_path = Path(relative_path)
    if data_path.is_absolute():
        return data_path
    candidate = notebook_dir / data_path
    if candidate.exists():
        return candidate.resolve()
    return (notebook_dir.parent / data_path).resolve()


In [ ]:
def load_otu_positives(data_path):
    file_path = resolve_data_path(data_path)
    if not file_path.exists():
        raise FileNotFoundError("No se encontro el archivo de datos configurado.")

    df = pd.read_csv(file_path)
    values = df.select_dtypes(include=[np.number]).to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]
    positives = values[values > 0]
    return positives, df.shape


def suggest_grid_size(n_values):
    if n_values <= 250_000:
        return 1000
    if n_values <= 1_000_000:
        return 1500
    return 2000


def make_log_grid(values, grid_size, bandwidth):
    positives = np.asarray(values, dtype=float)
    positives = positives[np.isfinite(positives)]
    positives = positives[positives > 0]
    if positives.size == 0:
        raise ValueError("Se requiere al menos un valor positivo.")
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")

    lower = max(float(np.min(positives)) * 1e-3, 1e-12)
    upper = float(np.max(positives) + 8.0 * bandwidth)
    return np.logspace(np.log10(lower), np.log10(upper), int(grid_size))


In [ ]:
KERNELS = ("gaussian", "epanechnikov", "tophat", "exponential", "linear", "cosine")
COLOR_MAP = {
    "gaussian": "#4c78a8",
    "epanechnikov": "#e45756",
    "tophat": "#f58518",
    "exponential": "#54a24b",
    "linear": "#b279a2",
    "cosine": "#9d755d",
}
FINITE_FAST_KERNELS = {"epanechnikov", "tophat", "linear", "cosine"}


def positive_mass(pdf, x_grid):
    return float(np.trapezoid(pdf, x_grid))


def normalize_conditional(pdf, x_grid):
    mass = positive_mass(pdf, x_grid)
    if not np.isfinite(mass) or mass <= 0:
        raise ValueError("La densidad KDE tiene masa no positiva o no finita.")
    return pdf / mass, mass


def mode_kde(pdf, x_grid):
    return float(x_grid[int(np.argmax(pdf))])


def evaluate_kde_reference(data, x_grid, bandwidth, kernel):
    kde = KernelDensity(kernel=kernel, bandwidth=bandwidth)
    kde.fit(np.asarray(data, dtype=float).reshape(-1, 1))
    return np.exp(kde.score_samples(np.asarray(x_grid, dtype=float).reshape(-1, 1)))


def evaluate_kde_gaussian_fast(data, x_grid, bandwidth):
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth, breadth_first=False)
    kde.fit(np.asarray(data, dtype=float).reshape(-1, 1))
    return np.exp(kde.score_samples(np.asarray(x_grid, dtype=float).reshape(-1, 1)))


def sorted_prefix(data):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    prefix_1 = np.concatenate(([0.0], np.cumsum(sorted_data)))
    prefix_2 = np.concatenate(([0.0], np.cumsum(sorted_data * sorted_data)))
    return sorted_data, prefix_1, prefix_2


def evaluate_finite_support_fast(data, x_grid, bandwidth, kernel):
    sorted_data, prefix_1, prefix_2 = sorted_prefix(data)
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    left = np.searchsorted(sorted_data, x - bandwidth, side="left")
    right = np.searchsorted(sorted_data, x + bandwidth, side="right")
    count = (right - left).astype(float)

    if kernel == "tophat":
        return 0.5 * count / (n * bandwidth)

    if kernel == "epanechnikov":
        sum_1 = prefix_1[right] - prefix_1[left]
        sum_2 = prefix_2[right] - prefix_2[left]
        sq_dist = count * x * x - 2.0 * x * sum_1 + sum_2
        return 0.75 * (count - sq_dist / (bandwidth * bandwidth)) / (n * bandwidth)

    if kernel == "linear":
        mid = np.searchsorted(sorted_data, x, side="right")
        mid = np.minimum(np.maximum(mid, left), right)
        left_count = (mid - left).astype(float)
        right_count = (right - mid).astype(float)
        left_sum = prefix_1[mid] - prefix_1[left]
        right_sum = prefix_1[right] - prefix_1[mid]
        abs_dist = x * left_count - left_sum + right_sum - x * right_count
        return (count - abs_dist / bandwidth) / (n * bandwidth)

    if kernel == "cosine":
        c = np.pi / (2.0 * bandwidth)
        prefix_cos = np.concatenate(([0.0], np.cumsum(np.cos(c * sorted_data))))
        prefix_sin = np.concatenate(([0.0], np.cumsum(np.sin(c * sorted_data))))
        sum_cos = prefix_cos[right] - prefix_cos[left]
        sum_sin = prefix_sin[right] - prefix_sin[left]
        values = np.cos(c * x) * sum_cos + np.sin(c * x) * sum_sin
        return (np.pi / 4.0) * values / (n * bandwidth)

    raise ValueError(f"Kernel no soportado por el metodo rapido: {kernel}")


def evaluate_exponential_fast(data, x_grid, bandwidth):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    order = np.argsort(x)
    xs = x[order]

    left_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = 0
    previous_x = xs[0]
    for i, current_x in enumerate(xs):
        if i > 0:
            acc *= np.exp(-(current_x - previous_x) / bandwidth)
        while data_idx < sorted_data.size and sorted_data[data_idx] <= current_x:
            acc += np.exp((sorted_data[data_idx] - current_x) / bandwidth)
            data_idx += 1
        left_sum[i] = acc
        previous_x = current_x

    right_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = sorted_data.size - 1
    previous_x = xs[-1]
    for i in range(xs.size - 1, -1, -1):
        current_x = xs[i]
        if i < xs.size - 1:
            acc *= np.exp(-(previous_x - current_x) / bandwidth)
        while data_idx >= 0 and sorted_data[data_idx] > current_x:
            acc += np.exp((current_x - sorted_data[data_idx]) / bandwidth)
            data_idx -= 1
        right_sum[i] = acc
        previous_x = current_x

    out_sorted = 0.5 * (left_sum + right_sum) / (n * bandwidth)
    out = np.empty_like(out_sorted)
    out[order] = out_sorted
    return out


def evaluate_kde_fast(data, x_grid, bandwidth, kernel):
    data = np.asarray(data, dtype=float).ravel()
    data = data[np.isfinite(data)]
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")
    if kernel not in KERNELS:
        raise ValueError(f"Kernel desconocido: {kernel}")
    if kernel in FINITE_FAST_KERNELS:
        return evaluate_finite_support_fast(data, x_grid, bandwidth, kernel)
    if kernel == "exponential":
        return evaluate_exponential_fast(data, x_grid, bandwidth)
    return evaluate_kde_gaussian_fast(data, x_grid, bandwidth)


def compare_kernel_to_reference(data, x_grid, bandwidth, kernel, atol=1e-10, rtol_peak=1e-8):
    candidate = evaluate_kde_fast(data, x_grid, bandwidth, kernel)
    reference = evaluate_kde_reference(data, x_grid, bandwidth, kernel)
    max_abs = float(np.max(np.abs(candidate - reference)))
    peak = max(float(np.max(np.abs(reference))), np.finfo(float).tiny)
    rel_peak = float(max_abs / peak)
    candidate_norm, _ = normalize_conditional(candidate, x_grid)
    reference_norm, _ = normalize_conditional(reference, x_grid)
    norm_max_abs = float(np.max(np.abs(candidate_norm - reference_norm)))
    return {
        "kernel": kernel,
        "bandwidth": bandwidth,
        "error_abs_max": max_abs,
        "error_relativo_pico": rel_peak,
        "diferencia_curva": norm_max_abs,
        "pasa_validacion": bool(max_abs <= atol or rel_peak <= rtol_peak),
    }


In [ ]:
# Configuracion
DATA_PATH = "../Datos/otu_data_converted.csv"
selected_method = "cv_por_kernel"
GRID_SIZE = 1000
VALIDATE_FAST_METHOD = True
VALIDATION_GRID_SIZE = 300

# Copiar estos valores desde Estimacion_Bandwidths.ipynb.
kernel_bandwidths = {
    "gaussian": 129.874938044,
    "epanechnikov": 1.77134956169,
    "tophat": 1.77134956169,
    "exponential": 15.167528295,
    "linear": 1.77134956169,
    "cosine": 1.77134956169,
}

# Pruebas particulares: solo afectan la figura individual de prueba.
test_kernel_bandwidths = {
    "gaussian": kernel_bandwidths["gaussian"],
    "epanechnikov": kernel_bandwidths["epanechnikov"],
    "tophat": kernel_bandwidths["tophat"],
    "exponential": kernel_bandwidths["exponential"],
    "linear": kernel_bandwidths["linear"],
    "cosine": kernel_bandwidths["cosine"],
}

values, data_shape = load_otu_positives(DATA_PATH)
x_grid = make_log_grid(values, GRID_SIZE, max(kernel_bandwidths.values()))

summary = pd.DataFrame([{
    "archivo": DATA_PATH,
    "filas": data_shape[0],
    "columnas": data_shape[1],
    "valores_positivos": len(values),
    "grid_size": GRID_SIZE,
}])
display(summary)


In [ ]:
bandwidth_df = pd.DataFrame(
    [{"kernel": kernel, "bandwidth": kernel_bandwidths[kernel]} for kernel in KERNELS]
)
display(bandwidth_df)


In [ ]:
reference_kernels = set()
validation_df = None

if VALIDATE_FAST_METHOD:
    validation_grid = make_log_grid(values, min(GRID_SIZE, VALIDATION_GRID_SIZE), max(kernel_bandwidths.values()))
    validation_rows = [
        compare_kernel_to_reference(values, validation_grid, kernel_bandwidths[kernel], kernel)
        for kernel in KERNELS
    ]
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)

    failed = validation_df.loc[~validation_df["pasa_validacion"], "kernel"].tolist()
    reference_kernels.update(failed)
    if failed:
        print(f"Kernels evaluados con referencia por seguridad: {failed}")
    else:
        print("Todos los kernels pasaron la validacion numerica.")


In [ ]:
if validation_df is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(validation_df["kernel"], validation_df["diferencia_curva"], color=[COLOR_MAP[k] for k in validation_df["kernel"]])
    ax.set_yscale("log")
    ax.set_xlabel("Kernel")
    ax.set_ylabel("Diferencia maxima")
    ax.set_title("Diferencia frente a la referencia")
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()


In [ ]:
densities = {}
summary_rows = []

start = time.perf_counter()
for kernel in KERNELS:
    bandwidth = kernel_bandwidths[kernel]
    if kernel in reference_kernels:
        pdf = evaluate_kde_reference(values, x_grid, bandwidth, kernel)
        metodo = "referencia"
    else:
        pdf = evaluate_kde_fast(values, x_grid, bandwidth, kernel)
        metodo = "rapido"

    density, mass = normalize_conditional(pdf, x_grid)
    densities[kernel] = density
    summary_rows.append({
        "kernel": kernel,
        "metodo": metodo,
        "bandwidth": bandwidth,
        "masa": mass,
        "moda_kde": mode_kde(density, x_grid),
    })
elapsed = time.perf_counter() - start

results_df = pd.DataFrame(summary_rows)
display(results_df)
print(f"Tiempo de evaluacion: {elapsed:.3f} s")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for kernel in KERNELS:
    ax.plot(
        x_grid,
        densities[kernel],
        color=COLOR_MAP[kernel],
        linewidth=1.8,
        label=f"{kernel} (h={kernel_bandwidths[kernel]:.3g})",
    )

ax.set_xscale("log")
ax.set_xlabel("Valor OTU positivo")
ax.set_ylabel("Densidad condicional")
ax.set_title("KDE de los seis kernels")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for ax, kernel in zip(axes.ravel(), KERNELS):
    ax.plot(x_grid, densities[kernel], color=COLOR_MAP[kernel], linewidth=1.8)
    ax.set_xscale("log")
    ax.set_title(f"{kernel} | h={kernel_bandwidths[kernel]:.3g}")
    ax.grid(True, alpha=0.25)

for ax in axes[-1, :]:
    ax.set_xlabel("Valor OTU positivo")
for ax in axes[:, 0]:
    ax.set_ylabel("Densidad condicional")

fig.suptitle("KDE por kernel", fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# Pruebas particulares por kernel.
# Modifica test_kernel_bandwidths en la configuracion para experimentar sin cambiar la evaluacion principal.
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for ax, kernel in zip(axes.ravel(), KERNELS):
    bandwidth = test_kernel_bandwidths[kernel]
    test_grid = make_log_grid(values, GRID_SIZE, bandwidth)
    pdf = evaluate_kde_fast(values, test_grid, bandwidth, kernel)
    density, _ = normalize_conditional(pdf, test_grid)
    ax.plot(test_grid, density, color=COLOR_MAP[kernel], linewidth=1.8)
    ax.set_xscale("log")
    ax.set_title(f"{kernel} | prueba h={bandwidth:.3g}")
    ax.grid(True, alpha=0.25)

for ax in axes[-1, :]:
    ax.set_xlabel("Valor OTU positivo")
for ax in axes[:, 0]:
    ax.set_ylabel("Densidad condicional")

fig.suptitle("Pruebas particulares de bandwidth por kernel", fontsize=13)
fig.tight_layout()
plt.show()
